# GSB 5544 — Topic 12.1: Project Exploration — NFL Running Backs and the Hall of Fame
*A 20-minute fill-in-and-discuss opener for the final project. One project idea taken from an outline to a running first version. Two maps keep it coherent: **ETV** (every step is an Extraction, a Transformation or a Visualization) and **SSVA** (every question gets a Summary Statistic, a Visualization and an Analysis sentence). Fill each `____` in the code and the prose, run the cells, and jot a sentence for each discussion prompt.*

## Part 0 — The idea in one paragraph

We want to know how **Hall of Fame running backs** compared with the *other* running backs who played at the same
time, and then use those comparisons to find **current and recently retired** players whose statistical profiles
look like a Hall of Famer's.

**The five project questions**

1. Which running backs, halfbacks and fullbacks are in the Pro Football Hall of Fame?
2. How did each Hall of Famer's rushing (and receiving) statistics compare with other running backs **in each season
   they played**?
3. What patterns show up across Hall of Fame careers: exceptional **peak** seasons, **sustained** performance, or both?
4. Which active or recently retired players show similar patterns at the same stage of their careers?
5. How much do the answers change with different statistics or a different **comparison group**?

**Key idea.** Compare a player with the other running backs in the same `____` *first*. A 1,200-yard season in a
year when the median qualified back gained 800 yards is 400 yards, or 50 %, above that season's median, and we can
also report the player's `____` among the qualifying backs that year. This makes comparisons more informative
across periods with different schedules and offensive styles. It does **not** remove every difference between eras.

## Part 1 — Two maps that make a project coherent

### ETV — every step is an Extraction, a Transformation or a Visualization

The project outline had thirteen steps. Label each one. (E = extract data from a source · T = transform it: clean,
join, filter, compute · V = visualize or tabulate for a reader.)

| # | Outline step | Stage |
|---|---|:---:|
| 1 | Set up the notebook: packages, data sources, analysis settings | — |
| 2 | Read the Wikipedia list of inductees and identify the running backs | `____` |
| 3 | Obtain player-season rushing and receiving statistics (permitted source) | `____` |
| 4 | Standardize columns and positions, resolve team changes, one row per player-season | `____` |
| 5 | Match each Hall of Famer to the correct player record (crosswalk) | `____` |
| 6 | Define each season's comparison group (≥ 50 carries) | `____` |
| 7 | Difference from the season median and percentile among contemporaries | `____` |
| 8 | Summarize Hall of Fame careers: typical, peak, longevity | `____` |
| 9 | Label players as inducted, active, retired-not-eligible, eligible | `____` |
| 10 | Compare active players with Hall of Famers at the same career stage | `____` |
| 11 | Tables and figures: within a season, across a career, current vs historical | `____` |
| 12 | Data checks, sensitivity checks, retrospective evaluation | `____` (and V) |
| 13 | Save results, state limitations, document next steps | — |

Three stages hide inside thirteen steps: **two** extractions, **eight** transformations, **one** visualization step
that should really be spread across the notebook, one figure per question. Steps 1 and 13 are bookkeeping.

### SSVA — every question is answered by a Summary Statistic, a Visualization, and an Analysis sentence

| Question | Summary statistic (the number) | Visualization (the figure) | Analysis (the sentence) |
|---|---|---|---|
| Q1 Who is in the Hall? | a count: 53 running backs, halfbacks and fullbacks | a table, not a chart | *N* Hall of Famers; *n* of them played in the seasons we can observe |
| Q2 Player vs that season | season `____` and the player's `____` | `____` of one season's rushing yards with the median and the player marked | "In 2006 Tomlinson gained 1,815 yards, 1,070 above the median" |
| Q3 Peak or sustained? | median season percentile; **share** of seasons at or above the 90th | `____` plot of percentile by career year; scatter of median vs share ≥ 90 | "Hall of Famers sit at about the 79th percentile in a typical season" |
| Q4 Who looks similar now? | the same two numbers for active players | the same scatter, active players highlighted | "Henry's profile sits inside the Hall of Fame cluster" |
| Q5 Does the answer move? | the same numbers under 25 / 50 / 100 carries | a sensitivity `____` | "The conclusion is stable" or "it depends on the threshold" |

A figure that does not sit in a row of this table does not belong in the notebook.

## Part 2 — E: Extract

Two sources, both permitted for this use, both **cached** so the notebook runs the same way every time:

| Need | Source | Cached file | What to watch |
|---|---|---|---|
| Hall of Fame names, positions, classes | Wikipedia, *List of Pro Football Hall of Fame inductees* (text CC BY-SA 4.0), accessed 2026-11-10 | `hof_inductees.csv` — 783 rows, one per inductee **and team** | repeated rows per person, mixed position labels, footnote marks in names |
| Season-by-season statistics | nflverse `stats_player_reg_<season>` releases, accessed 2026-11-10 | `nfl_rb_seasons_1999_2025.csv` — 4,619 RB / FB player-seasons, 27 seasons | coverage starts in `____`; a career that began earlier is only **partly** observed |

The outline's first choice, Pro Football Reference, restricts automated access and the use of its content for
machine-learning methods, so the first version uses nflverse and says so. Cache the extraction once (the code that
produced each file is in `tools/build_week12_project_exploration.py`), record the access date, and read the cache
in the notebook.

In [ ]:
import warnings
import pandas as pd
import numpy as np
from plotnine import *

warnings.filterwarnings("ignore")
DATA = "https://raw.githubusercontent.com/gato365/gsb5544_instructor_learn_prep/main/assignments/Data/"

LAST_COMPLETED_SEASON = 2025   # the 2026 season is in progress; compare completed seasons only
MIN_CARRIES = ____               # starting assumption for "a meaningful rushing role"; tested in Part 4

hof_raw = pd.read_csv(DATA + "hof_inductees.csv")
rb_raw  = pd.read_csv(DATA + "nfl_rb_seasons_1999_2025.csv")
print(hof_raw.shape, rb_raw.shape)
hof_raw.head()

The live extraction was one line, `pd.read_html(HOF_URL)`, followed by finding the right table **by its column
names** rather than by its index (the page has fifteen tables). The statistics file was built by reading
`stats_player_reg_<season>.csv` for each season from 1999 to 2025 and keeping the rows whose position is RB, HB or
FB.

In [ ]:
rb_raw.head()

In [ ]:
# One row per player per season?  The whole analysis assumes it.
assert not rb_raw.duplicated(["player_id", "season"]).any()
rb_raw.groupby("season")["player_id"].nunique().describe().round(0)

**Discuss.** Why cache an extraction instead of scraping every time the notebook runs?

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

## Part 3 — T: Transform

### 3.1 Q1 — Which inductees are running backs?  (clean the Hall of Fame list)

Three problems in the raw table: the position labels are mixed ("Halfback Pre-Modern Era: Two-Way Performer",
"Fullback", "Running back"); names carry footnote marks (`Marcus Allen** [4]`); and a person has one row **per
team**. First the trap: a careless filter.

In [ ]:
careless = hof_raw[hof_raw["Position"].str.contains("RB", case=False)]
careless["Position"].value_counts()

`"RB"` with `case=False` matches 127 rows, almost all of them `____` and `____`, because
both contain the letters *r-b*. Use the full words with word boundaries instead.

In [ ]:
is_rb = hof_raw["Position"].str.contains(____, case=False, regex=True)
hof_rb = hof_raw[is_rb].copy()

# strip footnote marks from the names:  "Marcus Allen** [4]" -> "Marcus Allen"
hof_rb["name"] = hof_rb["Inductee"].str.replace(____, "", regex=True).str.strip()

# first and last playing year from strings like "1993–1995", "2005, 2008", "1991"
years = hof_rb["Years"].str.findall(r"\d{4}")
hof_rb["first_year"] = years.map(lambda ys: min(map(int, ys)))
hof_rb["last_year"]  = years.map(lambda ys: max(map(int, ys)))

# one row per PERSON: earliest first year, latest last year, induction class
hof_people = (hof_rb.groupby(____)
                    .agg(hof_class=("Class", "min"), first_year=("first_year", "min"), last_year=("last_year", "max"),
                         position=("Position", "first"))
                    .reset_index()
                    .sort_values("first_year"))
print(len(hof_rb), "rows ->", len(hof_people), "people")
hof_people.tail(12)

**Q1 answered:** 53 Hall of Famers are listed at running back, halfback or fullback (17 of them as
pre-modern two-way players). Only **8** of them played in a season our statistics cover (1999 or later):
Thurman Thomas, Emmitt Smith, Jerome Bettis, Marshall Faulk, Curtis Martin, Terrell Davis, Edgerrin James, LaDainian Tomlinson. And only **2** of those — Edgerrin James and LaDainian Tomlinson — have their *whole* career inside the data.
Keep that number in mind for Part 5.

### 3.2 Crosswalk — connect each Hall of Famer to a `player_id`

Names suggest matches; they are not identifiers. Check: how many distinct `player_id` values does the name
*Adrian Peterson* have in the statistics?

In [ ]:
ids_per_name = rb_raw.groupby("player_display_name")["player_id"].nunique()
print(ids_per_name.sort_values(ascending=False).head(5))

# match the Hall of Famers whose careers reach 1999, by display name, then CHECK the playing years
observable = hof_people[hof_people["last_year"] >= 1999]
xwalk = (observable.merge(rb_raw.groupby(["player_id", "player_display_name"])["season"].agg(["min", "max"]).reset_index(),
                          left_on="name", right_on=____, how="left")
                   .rename(columns={"min": "first_observed", "max": "last_observed"}))
xwalk["coverage"] = np.where(xwalk["first_year"] >= 1999, "complete", "partial (pre-1999 seasons missing)")
xwalk[["name", "hof_class", "first_year", "last_year", "player_id", "first_observed", "last_observed", "coverage"]]

Two players share the name Adrian Peterson (2 ids: the Bears back of the 2000s and the Vikings back), and two
share Ricky Williams. A join on `name` alone would have merged two careers into one. Every match above was checked against the playing years, and
each Hall of Famer whose career started before 1999 is labelled **partial** rather than silently treated as a short
career.

In [ ]:
rb = rb_raw.merge(xwalk[["player_id", "hof_class", "coverage"]], on="player_id", how=____)
rb["hof"] = rb["hof_class"].notna()
rb["career_year"] = rb["season"] - rb.groupby("player_id")["season"].transform(____) + 1
print(rb["hof"].sum(), "Hall of Fame player-seasons out of", len(rb))

### 3.3 Q2 — Each season's comparison group, median and percentile

A qualified season has at least `____` carries. Within each season, the **median** rushing yards of the qualified backs
is the reference point; a player's **percentile** is the share of qualified backs at or below their total.

In [ ]:
qualified = rb.loc[(rb["season"] <= LAST_COMPLETED_SEASON) & (rb["carries"] >= ____)].copy()

by_season = qualified.groupby("season")["rushing_yards"]
qualified["season_median"]     = by_season.transform(____)
qualified["yards_above_median"] = qualified["rushing_yards"] - qualified["season_median"]
qualified["pct"]               = 100 * by_season.rank(method="max", ____)   # % of peers at or below, ties included

peers = qualified.groupby("season").size()
print(f"{len(qualified):,} qualified player-seasons; {peers.min()}–{peers.max()} peers per season")
qualified.loc[qualified["player_display_name"] == "LaDainian Tomlinson",
              ["season", "career_year", "carries", "rushing_yards", "season_median", "yards_above_median", "pct"]].round(0)

**Q2 for one player-season:** in 2006 Tomlinson gained 1,815 rushing yards when the median of the 59
qualified backs was 745 — 1,070 yards above the median, 2.4 times the median, the
`____` percentile.

### 3.4 Q3 — Career summaries: typical season, best season, how often exceptional

One row per player. The two numbers the whole project rests on are the **median** season percentile (sustained) and
the **share** of qualified seasons at or above the 90th percentile (peak).

In [ ]:
career = (qualified.groupby("player_id")
                   .agg(player=("player_display_name", "first"), hof=("hof", "first"), coverage=("coverage", "first"),
                        n_qualified=("season", "size"), first_q=("season", "min"), last_q=("season", "max"),
                        median_pct=("pct", ____), max_pct=("pct", "max"),
                        share_90=("pct", lambda s: ____),
                        rush_yards=("rushing_yards", "sum"))
                   .reset_index())
career["status"] = np.select([career["hof"], career["last_q"] == LAST_COMPLETED_SEASON],
                             ["Hall of Fame", "active (played 2025)"], "not seen in 2025")
career.loc[career["hof"]].sort_values("median_pct", ascending=False).round(1)

Only 7 of the 8 observable Hall of Famers have a row: Thurman Thomas never reached 50 carries in
an observed season. A percentile summary of a *partial* career describes its tail (Davis: three seasons after the
knee injury), not the player — which is why `coverage` travels with every row.

Status is **not** retirement: a player absent from 2025 may be injured, unsigned or retired, and a retired player who
is eligible but not inducted is not a confirmed "no". The outline's step 9 (a roster or retirement source) is still
to do; until then the label says exactly what the data say.

## Part 4 — V: Visualize, one figure per question

Colors are fixed by **entity**, never by rank: Hall of Famers orange, active players blue, everyone else gray. The
caption of each figure names the question it answers and the statistic it shows.

### Figure 1 (Q2) — One season: where did Tomlinson sit among the 2006 backs?
*Statistic: the season median and one player's percentile. Form: a histogram of the 59 qualified backs' rushing yards.*

In [ ]:
HOF, ACTIVE, OTHER = "#eb6834", "#2a78d6", "#9a9892"

s06 = qualified[qualified["season"] == ____]
lt  = s06[s06["player_display_name"] == "LaDainian Tomlinson"].iloc[0]
med = s06["season_median"].iloc[0]
top = np.histogram(s06["rushing_yards"], bins=np.arange(0, s06["rushing_yards"].max() + 100, 100))[0].max()

fig1 = (ggplot(s06, aes(x="rushing_yards"))
        + geom_histogram(binwidth=100, boundary=0, fill=OTHER, color="white")
        + geom_vline(xintercept=____, linetype="dashed")
        + annotate("text", x=med + 30, y=top, label=f"season median {med:,.0f}", ha="left", size=9)
        + geom_vline(xintercept=lt["rushing_yards"], color=HOF, size=1.2)
        + annotate("text", x=lt["rushing_yards"] - 30, y=top, label=f"Tomlinson {lt['rushing_yards']:,.0f}  ({lt['pct']:.0f}th pct)",
                   ha="right", color=HOF, size=9)
        + labs(x="Rushing yards, 2006 regular season", y="Qualified running backs",
               title=f"2006: {len(s06)} backs with at least {MIN_CARRIES} carries")
        + theme_minimal() + theme(figure_size=(8, 3.8)))
fig1

**Interpret Figure 1.** Write the analysis sentence this figure supports: name the statistic, the comparison and
the conclusion. What does the dashed line add that the orange line alone would not? Is this a *peak* or a
*sustained* claim?

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

### Figure 2 (Q3) — Careers: percentile by career year, one line per player
*Statistic: the season percentile each year. Form: `____` plot, career year on the horizontal axis, one line per player.*

**You choose the players.** Career year 1 must be real. For the Hall of Famers, Wikipedia's playing years say
whether it is (`coverage == "complete"`); for everyone else the only safe rule is a first season of 2000 or later
(a 1999 start could be a career already in progress). The table below lists the candidates: pick the two complete
Hall of Fame careers plus **two players of your own** — one retired and one active makes the clearest picture —
and make sure your neighbour picks different ones.

In [ ]:
first_seen = rb.groupby("player_id")["season"].min().rename("first_seen")
candidates = (career.merge(first_seen, on="player_id")
                    .query("(first_seen >= 2000 or coverage == 'complete') and n_qualified >= 5")
                    .sort_values("median_pct", ascending=False))
print(len(candidates), "players with a real career year 1 and at least five qualified seasons")
candidates[["player", "status", "n_qualified", "first_q", "last_q", "median_pct", "share_90"]].head(25).round(1)

In [ ]:
show = ____   # two Hall of Famers + two of your choice

paths = qualified[qualified["player_display_name"].isin(show) & (qualified["coverage"] != "partial (pre-1999 seasons missing)")].copy()
paths["who"] = np.select([paths["hof"], paths["season"] == LAST_COMPLETED_SEASON], ["Hall of Fame", "active"], "not seen in 2025")
paths["who"] = paths.groupby("player_id")["who"].transform("last")
ends = paths.sort_values("career_year").groupby("player_id").tail(1)

fig2 = (ggplot(paths, aes(x="career_year", y="pct", group="player_id", color="who"))
        + geom_hline(yintercept=90, linetype="dotted")
        + geom_line(size=1) + geom_point(size=2)
        + geom_text(ends, aes(label="player_display_name"), ha="left", nudge_x=0.3, size=9, show_legend=False)
        + scale_color_manual(values={"Hall of Fame": HOF, "active": ACTIVE, "not seen in 2025": "#52514e"}, name="")
        + scale_x_continuous(breaks=range(1, 18), limits=(1, 19))
        + labs(x="Career year (qualified seasons only)", y="Rushing-yard percentile among that season's backs",
               color="", title="Season percentile by career year: " + ", ".join(show))
        + theme_minimal() + theme(figure_size=(9, 4.2), legend_position="top"))
fig2

**Interpret Figure 2.** For the players *you* chose: which lines make a **peak** argument (years above the dotted
90th line) and which a **sustained** one (many years near it)? Where does each line start, and what does a gap in a
line mean? Write the analysis sentence, naming your players.

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

### Figure 3 (Q3 and Q4) — Sustained vs peak for every back with at least five qualified seasons
*Statistics: median season percentile (x) and share of seasons at or above the 90th percentile (y). Form: scatter;
Hall of Famers and active players highlighted and labelled.*

In [ ]:
five = career[career["n_qualified"] >= ____].copy()
five["coverage"] = five["coverage"].fillna("complete")
# selective direct labels: the Hall of Famers and four comparison players; names placed left or right to avoid collisions
named = five[five["hof"] | five["player"].isin(["Derrick Henry", "Christian McCaffrey", "Frank Gore", "Adrian Peterson"])]
to_left = named[named["player"].isin(["Adrian Peterson", "Curtis Martin", "Edgerrin James", "Jerome Bettis", "Derrick Henry"])]
to_right = named.drop(to_left.index)

fig3 = (ggplot(five, aes(x="median_pct", y="share_90"))
        + geom_point(five[five["status"] == "not seen in 2025"], color=OTHER, alpha=0.6, size=2)
        + geom_point(five[five["status"] != "not seen in 2025"], aes(color="status", shape="coverage"), size=3)
        + geom_text(to_right, aes(label="player"), size=7, ha="left", nudge_x=1, color="#0b0b0b")
        + geom_text(to_left, aes(label="player"), size=7, ha="right", nudge_x=-1, color="#0b0b0b")
        + scale_color_manual(values={"Hall of Fame": HOF, "active (played 2025)": ACTIVE})
        + scale_shape_manual(values={"complete": "o", "partial (pre-1999 seasons missing)": "^"})
        + scale_y_continuous(labels=lambda l: [f"{100*v:.0f}%" for v in l])
        + labs(x="Median season percentile (sustained)", y="Share of seasons at or above the 90th percentile (peak)",
               color="", shape="", title=f"{len(five)} running backs with at least five qualified seasons, 1999–2025")
        + theme_minimal() + theme(figure_size=(9, 5.2), legend_position="top"))
fig3

**Interpret Figure 3.** Where do the orange points sit, and what do the two axes say about them? What are the
triangles, and why are two of them low? Then Q4: which blue points sit inside the Hall of Fame cluster, and what
exactly is being claimed about them?

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

In [ ]:
cols = ["player", "status", "coverage", "n_qualified", "first_q", "last_q", "median_pct", "max_pct", "share_90", "rush_yards"]
(five[five["status"] != "Hall of Fame"]
     .sort_values(____, ascending=False)
     .head(12)[cols].round(1))

The candidate table sorts on **one** clearly stated measure and shows the others beside it rather than hiding them
in a single score.

### Figure 4 (Q5) — Does the answer move with the carry threshold?
*Statistic: the Hall of Fame group's median percentile and share ≥ 90 under 25, 50 and 100 carries. Form: a table.*

In [ ]:
rows = []
for thr in ____:
    q = rb[(rb["season"] <= LAST_COMPLETED_SEASON) & (rb["carries"] >= thr)].copy()
    q["pct"] = 100 * q.groupby("season")["rushing_yards"].rank(method="max", pct=True)
    c = q.groupby("player_id").agg(hof=("hof", "first"), median_pct=("pct", "median"),
                                   share_90=("pct", lambda s: (s >= 90).mean()))
    rows.append({"min_carries": thr, "peers_per_season": int(q.groupby("season").size().median()),
                 "hof_median_pct": c[c["hof"]]["median_pct"].median(),
                 "hof_share_90": c[c["hof"]]["share_90"].median(),
                 "others_median_pct": c[~c["hof"]]["median_pct"].median()})
pd.DataFrame(rows).round(2)

**Interpret Figure 4 (the table).** Read down each column: what happens to the peer group, to the Hall of Fame
median, and to the other backs' median as the threshold rises? Why does one move and the other not? Then say in
one sentence whether the Q3 conclusion is stable.

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

## Part 5 — How do we improve this project?

Work through these in pairs. Each one is a gap in the first version that the SSVA table or the ETV map makes visible.

**5.1 Coverage.** Only 2 of 53 Hall of Fame running backs have complete careers in the data. What does
that do to Q3 and Q4, and what are the two honest ways forward?

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

**5.2 The comparison group.** The 50-carry rule defines "a running back". Who disappears, and which project question
suffers?

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

**5.3 Status.** "Not seen in 2025" is doing a lot of work. What is wrong with it, and what source fixes it?

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

**5.4 What the statistics cannot see.** Name two things voters weigh that no column in `rb` contains. Could they be
extracted?

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

**5.5 Why not fit a classifier now?** We have a labelled column (`hof`) and features. What stops a model from being
the next cell?

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

**5.6 Coherence check.** Go back to the SSVA table in Part 1. Is there a cell in this notebook that does not sit in
one of its rows? Is there a row with no figure?

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

---
## For your own final project — the checklist

1. **One paragraph, then numbered questions.** If a question cannot be numbered it is not a question yet.
2. **ETV.** Label every planned step E, T or V. Cache every E with its access date and the code that made it.
   Write every T as a named step with a check (`assert` one row per key; print the counts you expect).
3. **SSVA.** For each question, name the **summary statistic** before choosing the **visualization**, and write the
   **analysis sentence** the figure is supposed to support. One figure per question; colors fixed by entity.
4. **Say what the data cannot answer** (coverage, definitions, status) in the notebook, not in the presentation's
   last slide.
5. **Sensitivity before modelling.** Change the one rule you chose (the threshold, the peer group) and report what
   moves. A model comes after the descriptive version is coherent, and only with data licensed for it.